# Colab Starter: Class Student Accent Training

This notebook runs the class-data pipeline fully in Colab: Drive mount, repo sync, metadata prep, speaker-disjoint splits, validation, feature extraction, and training.

In [ ]:
import shlex
import subprocess
from pathlib import Path

def run(cmd, cwd=None):
    print(f'\n$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def q(path_like):
    return shlex.quote(str(path_like))

REPO_URL = 'https://github.com/Saikumarlingaraju/Audio.git'
REPO_BRANCH = 'main'
PROJECT_DIR = Path('/content/iiitpro')

# Audio root in Drive must contain state folders directly (andhra_pradesh/, telangana/, ...).
DRIVE_AUDIO_ROOT = Path('/content/drive/MyDrive/iiitpro_audio/organized_by_state_clean')

RUN_HUBERT = True
RUN_ROBUST = True
BASELINE_EPOCHS = 20
ROBUST_EPOCHS = 30

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped (not in Colab?):', exc)

run('nvidia-smi || true')
run('apt-get update && apt-get install -y ffmpeg')
run('python -m pip install --upgrade pip')

if PROJECT_DIR.exists():
    run('git -C {repo} fetch --all'.format(repo=q(PROJECT_DIR)))
    run('git -C {repo} checkout {branch}'.format(repo=q(PROJECT_DIR), branch=REPO_BRANCH))
    run('git -C {repo} pull origin {branch}'.format(repo=q(PROJECT_DIR), branch=REPO_BRANCH))
else:
    run('git clone --branch {branch} {url} {dest}'.format(branch=REPO_BRANCH, url=REPO_URL, dest=q(PROJECT_DIR)))

run('pip install -r requirements.txt', cwd=str(PROJECT_DIR))

In [ ]:
CLASS_DATA_DIR = PROJECT_DIR / 'data/collections/class_students/organized_by_state_clean'
METADATA_ALL = CLASS_DATA_DIR / 'metadata_all.csv'
METADATA_READY = CLASS_DATA_DIR / 'metadata_training_ready.csv'
SPLIT_DIR = PROJECT_DIR / 'data/splits/class_students'
MFCC_FEATURE_DIR = PROJECT_DIR / 'data/features/class_students/mfcc'
HUBERT_FEATURE_DIR = PROJECT_DIR / 'data/features/class_students/hubert'

if not METADATA_ALL.exists():
    raise FileNotFoundError(f'Metadata not found in repo: {METADATA_ALL}')
if not DRIVE_AUDIO_ROOT.exists():
    raise FileNotFoundError(f'Drive audio folder missing: {DRIVE_AUDIO_ROOT}')

print('Metadata source:', METADATA_ALL)
print('Drive audio root:', DRIVE_AUDIO_ROOT)

In [ ]:
run(
    'python scripts/prepare_class_metadata_for_training.py ' +
    '--metadata {metadata} --audio_root {audio_root} --output {output} --min_bytes 1000 --strict'.format(
        metadata=q(METADATA_ALL),
        audio_root=q(DRIVE_AUDIO_ROOT),
        output=q(METADATA_READY),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python src/data/create_splits.py ' +
    '--metadata {metadata} --output_dir {output_dir} --train_ratio 0.7 --dev_ratio 0.15 --test_ratio 0.15'.format(
        metadata=q(METADATA_READY),
        output_dir=q(SPLIT_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python scripts/validate_class_dataset.py ' +
    '--metadata {metadata} --audio_root {audio_root} --split_dir {split_dir}'.format(
        metadata=q(METADATA_READY),
        audio_root=q(DRIVE_AUDIO_ROOT),
        split_dir=q(SPLIT_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

In [ ]:
run(
    'python src/features/mfcc_extractor.py ' +
    '--metadata {metadata} --audio_dir {audio_dir} --output_dir {output_dir}'.format(
        metadata=q(METADATA_READY),
        audio_dir=q(DRIVE_AUDIO_ROOT),
        output_dir=q(MFCC_FEATURE_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python train_simple.py ' +
    '--features_path {features} --train_csv {train_csv} --val_csv {val_csv} --test_csv {test_csv} ' +
    '--output_dir {output_dir} --num_epochs {epochs} --batch_size 8'.format(
        features=q(MFCC_FEATURE_DIR / 'mfcc_stats.pkl'),
        train_csv=q(SPLIT_DIR / 'train.csv'),
        val_csv=q(SPLIT_DIR / 'dev.csv'),
        test_csv=q(SPLIT_DIR / 'test.csv'),
        output_dir=q(PROJECT_DIR / 'experiments/class_students_colab_mfcc'),
        epochs=BASELINE_EPOCHS,
    ),
    cwd=str(PROJECT_DIR),
)

In [ ]:
if RUN_HUBERT:
    run(
        'python src/features/hubert_extractor.py ' +
        '--metadata {metadata} --audio_dir {audio_dir} --output_dir {output_dir} --extract_layer 3 --pooling mean'.format(
            metadata=q(METADATA_READY),
            audio_dir=q(DRIVE_AUDIO_ROOT),
            output_dir=q(HUBERT_FEATURE_DIR),
        ),
        cwd=str(PROJECT_DIR),
    )

if RUN_ROBUST:
    run(
        'python train_robust.py ' +
        '--features_path {features} --train_csv {train_csv} --val_csv {val_csv} --test_csv {test_csv} ' +
        '--output_dir {output_dir} --model_type robust_mlp --num_epochs {epochs} --batch_size 8'.format(
            features=q(HUBERT_FEATURE_DIR / 'hubert_layer3_mean.pkl'),
            train_csv=q(SPLIT_DIR / 'train.csv'),
            val_csv=q(SPLIT_DIR / 'dev.csv'),
            test_csv=q(SPLIT_DIR / 'test.csv'),
            output_dir=q(PROJECT_DIR / 'experiments/class_students_colab_hubert_robust'),
            epochs=ROBUST_EPOCHS,
        ),
        cwd=str(PROJECT_DIR),
    )

In [ ]:
RUN_EXPORT_DIR = Path('/content/drive/MyDrive/iiitpro_runs/class_students')
RUN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run('cp -r {src} {dst} || true'.format(src=q(PROJECT_DIR / 'experiments/class_students_colab_mfcc'), dst=q(RUN_EXPORT_DIR)))
run('cp -r {src} {dst} || true'.format(src=q(PROJECT_DIR / 'experiments/class_students_colab_hubert_robust'), dst=q(RUN_EXPORT_DIR)))
run('cp -r {src} {dst} || true'.format(src=q(SPLIT_DIR), dst=q(RUN_EXPORT_DIR)))

print('Artifacts copied to:', RUN_EXPORT_DIR)